In [ ]:
# ---- Configuration: edit config/config.yaml (see README) ----
# CSF: genetic groups (MutBpos, MutCpos, NoMut) and longitudinal DrugA/DrugB samples, PCA, expected correlations
# Input files are read from, and outputs written to, <longitudinal_workdir> (see README for the file list).
import os, sys
from pathlib import Path
_root = Path(os.environ.get("PROJECT_ROOT") or next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "config.py").exists()))
os.environ["PROJECT_ROOT"] = str(_root)
sys.path.insert(0, str(_root))
from utils.config import cfg, cfg_path
os.chdir(cfg_path("longitudinal_workdir", create=True))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.spatial.distance import euclidean, squareform
from scipy.cluster import hierarchy
import textwrap
from scipy.spatial.distance import euclidean
from scipy.cluster.hierarchy import linkage, dendrogram  # Add this import statement
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from matplotlib.patches import Patch

In [ ]:
Data = pd.read_csv("Data_final_analysis.csv", index_col=0)
Data.head(7000000)

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
filtered_df = Data.loc[:, ~Data.columns.str.contains('DrugA-12months|DrugB')]
filtered_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(filtered_df)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(filtered_df.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(50, 20))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(filtered_df.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(filtered_df.columns) + 1))
#ax.grid(True)

plt.tight_layout()
plt.savefig('Eigen_values_Bar_grah.png')

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutCpos_baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutCpos_BL{i+1}' if col in mutCpos_baseline_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

column_colors = {
    col: 'DimGray' if col in control_cols else
         'MediumVioletRed' if col in mutCpos_baseline_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutCpos_baseline_cols[0]: 'MutCpos Baseline',
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutCpos_baseline_cols, mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutCpos_baseline_cols[0]: 'MediumVioletRed',
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['MediumVioletRed', '#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutCpos_baseline_cols:
        marker = 's'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutCpos_baseline_cols:
        marker = 's'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutCpos_baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutCpos_BL{i+1}' if col in mutCpos_baseline_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

column_colors = {
    col: 'DimGray' if col in control_cols else
         'MediumVioletRed' if col in mutCpos_baseline_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutCpos_baseline_cols[0]: 'MutCpos Baseline',
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutCpos_baseline_cols, mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutCpos_baseline_cols[0]: 'MediumVioletRed',
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['MediumVioletRed', '#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutCpos_baseline_cols:
        marker = 's'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
         #    weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutCpos_baseline_cols:
        marker = 's'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
       #               facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC2 centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Copy data
Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutCpos_baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutCpos_BL{i+1}' if col in mutCpos_baseline_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

column_colors = {
    col: 'DimGray' if col in control_cols else
         'MediumVioletRed' if col in mutCpos_baseline_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Legend setup
selected_legend_columns = {
    mutCpos_baseline_cols[0]: 'MutCpos Baseline',
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Groups for centroids
selected_columns_list = [mutCpos_baseline_cols, mutB_cols, noMut_cols, control_cols]
centroid_colors = {
    mutCpos_baseline_cols[0]: 'MediumVioletRed',
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}
outline_colors = ['MediumVioletRed', '#800080', 'green', 'DimGray']

# Standardize data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA with 3 components for PC3 access
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = ('o' if col in control_cols else
              's' if col in mutCpos_baseline_cols else
              'D' if col in mutB_cols else
              '^' if col in noMut_cols else 'x')
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend entries
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = ('o' if col in control_cols else
              's' if col in mutCpos_baseline_cols else
              'D' if col in mutB_cols else
              '^' if col in noMut_cols else 'x')
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # PC1 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Labels and display
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples (PC1 vs PC3) with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC3.png')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Copy data
Data_csf = filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutCpos_baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutCpos_BL{i+1}' if col in mutCpos_baseline_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'MediumVioletRed' if col in mutCpos_baseline_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Legend setup
selected_legend_columns = {
    mutCpos_baseline_cols[0]: 'MutCpos Baseline',
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Groups for centroids
selected_columns_list = [mutCpos_baseline_cols, mutB_cols, noMut_cols, control_cols]
centroid_colors = {
    mutCpos_baseline_cols[0]: 'MediumVioletRed',
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}
outline_colors = ['MediumVioletRed', '#800080', 'green', 'DimGray']

# Standardize data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA with 3 components for PC3 access
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = ('o' if col in control_cols else
              's' if col in mutCpos_baseline_cols else
              'D' if col in mutB_cols else
              '^' if col in noMut_cols else 'x')
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
     #        weight='bold', color='black', ha='center', va='center')

# Legend entries
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = ('o' if col in control_cols else
              's' if col in mutCpos_baseline_cols else
              'D' if col in mutB_cols else
              '^' if col in noMut_cols else 'x')
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # PC1 vs PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Labels and display
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples (PC1 vs PC3) with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC3 centroids.png')
plt.show()


In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
pc_loadings_df = pd.DataFrame(pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=Data_csf.index)

# Save to CSV
pc_loadings_df.to_csv('pc_loadings.csv')

# Display the loadings
pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc1_sorted_df = pd.DataFrame(pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
pc1_loading_df_sorted = pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(pc1_loading_df_sorted.index, pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('All PC1 loadings.png')
plt.show()


In [ ]:
# Extract only PC2 loadings from the full loadings DataFrame
pc2_sorted_df = pd.DataFrame(pc_loadings_df['PC2'])

# Sort genes by their PC2 loading in descending order
pc2_loading_df_sorted = pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC2 loadings to CSV
pc2_loading_df_sorted.to_csv('pc2_loading_df_sorted.csv')

# Display the sorted DataFrame
pc2_loading_df_sorted

In [ ]:
# Define custom colormap for PC2
custom_cmap_pc2 = mcolors.TwoSlopeNorm(vmin=pc2_loading_df_sorted['PC2'].min(),
                                       vcenter=0,
                                       vmax=pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap_pc2(pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(pc2_loading_df_sorted.index, pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc2)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('All PC2 loadings.png')
plt.show()

In [ ]:
# Extract only PC3 loadings from the full loadings DataFrame
pc3_sorted_df = pd.DataFrame(pc_loadings_df['PC3'])

# Sort genes by their PC3 loading in descending order
pc3_loading_df_sorted = pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Save the sorted PC3 loadings to CSV
pc3_loading_df_sorted.to_csv('pc3_loading_df_sorted.csv')

# Display the sorted DataFrame
pc3_loading_df_sorted

In [ ]:
# Define custom colormap for PC3
custom_cmap_pc3 = mcolors.TwoSlopeNorm(vmin=pc3_loading_df_sorted['PC3'].min(),
                                       vcenter=0,
                                       vmax=pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap_pc3(pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(pc3_loading_df_sorted.index, pc3_loading_df_sorted['PC3'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc3)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('All PC3 loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC1' values in descending order
pc1_loading_df_sorted = pc1_loading_df_sorted.sort_values(by='PC1', ascending=False)

# Extract the top 20 and bottom 20 genes
top_20_genes_PC1 = pc1_loading_df_sorted.head(20)
bottom_20_genes_PC1 = pc1_loading_df_sorted.tail(20)

# Combine both into one DataFrame for convenience
top_bottom_20_genes_PC1 = pd.concat([top_20_genes_PC1, bottom_20_genes_PC1])

# Save the results to a CSV file
top_bottom_20_genes_PC1.to_csv('top_bottom_20_genes_PC1.csv')
top_bottom_20_genes_PC1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC1 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=top_bottom_20_genes_PC1['PC1'].min(),
                                   vcenter=0,
                                   vmax=top_bottom_20_genes_PC1['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(top_bottom_20_genes_PC1['PC1']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(top_bottom_20_genes_PC1.index, top_bottom_20_genes_PC1['PC1'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('PC1 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC1 Loadings')

# Show the plot
plt.tight_layout()
plt.savefig('PC1_loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC2' values in descending order
pc2_loading_df_sorted = pc2_loading_df_sorted.sort_values(by='PC2', ascending=False)

# Extract the top 20 and bottom 20 genes
top_20_genes_PC2 = pc2_loading_df_sorted.head(20)
bottom_20_genes_PC2 = pc2_loading_df_sorted.tail(20)

# Combine top and bottom genes
top_bottom_20_genes_PC2 = pd.concat([top_20_genes_PC2, bottom_20_genes_PC2])

# Save to CSV
top_bottom_20_genes_PC2.to_csv('top_bottom_20_genes_PC2.csv')

# Display
top_bottom_20_genes_PC2

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC2 with center at zero
custom_cmap_pc2 = mcolors.TwoSlopeNorm(vmin=top_bottom_20_genes_PC2['PC2'].min(),
                                       vcenter=0,
                                       vmax=top_bottom_20_genes_PC2['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define bar colors based on PC2 values
colors = plt.cm.RdBu_r(custom_cmap_pc2(top_bottom_20_genes_PC2['PC2']))

# Create horizontal bar plot using index as gene names
bars = ax.barh(top_bottom_20_genes_PC2.index, top_bottom_20_genes_PC2['PC2'], color=colors)

# Invert y-axis
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc2)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('PC2 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC2 Loadings')

# Save and show plot
plt.tight_layout()
plt.savefig('PC2_loadings.png')
plt.show()

In [ ]:
# Sort the DataFrame by 'PC3' values in descending order
pc3_loading_df_sorted = pc3_loading_df_sorted.sort_values(by='PC3', ascending=False)

# Extract the top 20 and bottom 20 genes
top_20_genes_PC3 = pc3_loading_df_sorted.head(20)
bottom_20_genes_PC3 = pc3_loading_df_sorted.tail(20)

# Combine top and bottom genes
top_bottom_20_genes_PC3 = pd.concat([top_20_genes_PC3, bottom_20_genes_PC3])

# Save to CSV
top_bottom_20_genes_PC3.to_csv('top_bottom_20_genes_PC3.csv')

# Display
top_bottom_20_genes_PC3

In [ ]:
# Define custom colormap for PC3 with center at zero
custom_cmap_pc3 = mcolors.TwoSlopeNorm(vmin=top_bottom_20_genes_PC3['PC3'].min(),
                                       vcenter=0,
                                       vmax=top_bottom_20_genes_PC3['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define bar colors based on PC3 values
colors = plt.cm.RdBu_r(custom_cmap_pc3(top_bottom_20_genes_PC3['PC3']))

# Create horizontal bar plot using index as gene names
bars = ax.barh(top_bottom_20_genes_PC3.index, top_bottom_20_genes_PC3['PC3'], color=colors)

# Invert y-axis
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc3)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('PC3 Loadings')
ax.set_ylabel('Proteins')
ax.set_title('PC3 Loadings')

# Save and show plot
plt.tight_layout()
plt.savefig('PC3_loadings.png')
plt.show()

In [ ]:
# List of desired gene/protein names
target_names = ['UCHL1', 'GOT1', 'NEFL', 'NEFH', 'NPY', 'AÎ²42']

# Filter the DataFrame
subset_df = filtered_df.loc[filtered_df.index.isin(target_names)]
subset_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = subset_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutCpos_baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutCpos_BL{i+1}' if col in mutCpos_baseline_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'MediumVioletRed' if col in mutCpos_baseline_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutCpos_baseline_cols[0]: 'MutCpos Baseline',
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutCpos_baseline_cols, mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutCpos_baseline_cols[0]: 'MediumVioletRed',
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['MediumVioletRed', '#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutCpos_baseline_cols:
        marker = 's'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutCpos_baseline_cols:
        marker = 's'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC2 Sigs prots.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = subset_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutCpos_baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutCpos_BL{i+1}' if col in mutCpos_baseline_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'MediumVioletRed' if col in mutCpos_baseline_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutCpos_baseline_cols[0]: 'MutCpos Baseline',
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutCpos_baseline_cols, mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutCpos_baseline_cols[0]: 'MediumVioletRed',
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['MediumVioletRed', '#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutCpos_baseline_cols:
        marker = 's'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
         #    weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutCpos_baseline_cols:
        marker = 's'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
   # ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                    #  facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PC1 VS PC2 Sigs prots Centroids.png')
plt.show()

In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
pc_loadings_df = pd.DataFrame(pc_loadings.T, columns=['PC1', 'PC2'], index=Data_csf.index)

# Save to CSV
#pc_loadings_df.to_csv('pc_loadings.csv')

# Display the loadings
pc_loadings_df

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC1 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc_loadings_df['PC1'].min(),
                                   vcenter=0,
                                   vmax=pc_loadings_df['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(pc_loadings_df['PC1']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(pc_loadings_df.index, pc_loadings_df['PC1'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('PC1 Loadings')
ax.set_ylabel('Gene Names')
ax.set_title('Significant proteins PC1 Loadings')

# Show the plot
plt.tight_layout()
plt.savefig('Significant_proteins_PC1_loadings.png')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define custom colormap for PC2 with vcenter set to zero
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc_loadings_df['PC2'].min(),
                                   vcenter=0,
                                   vmax=pc_loadings_df['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(10, 10))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(pc_loadings_df['PC2']))

# Create the horizontal bar plot using the index (gene names) as labels
bars = ax.barh(pc_loadings_df.index, pc_loadings_df['PC2'], color=colors)

# Invert the y-axis to have the top gene at the top
ax.invert_yaxis()

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('PC2 Loadings')
ax.set_ylabel('Gene Names')
ax.set_title('Significant Proteins PC2 Loadings')

# Show the plot
plt.tight_layout()
plt.savefig('Significant_proteins_PC2_loadings.png')
plt.show()

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
MutCpos_filtered_df = Data.loc[:, ~Data.columns.str.contains('Control|MutB|NoMut|DrugB')]
MutCpos_filtered_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Standardize the log-transformed data
std_scaler = StandardScaler()
data_standardized = std_scaler.fit_transform(MutCpos_filtered_df)

# Transpose the data for PCA along columns
data_transposed = data_standardized.T

# Create an instance of PCA with more components
pca_all = PCA(n_components=len(MutCpos_filtered_df.columns))

# Fit the PCA model to the transposed data (compute principal components along columns)
pca_all.fit(data_transposed)

# Calculate the explained variance ratio (as a percentage) for each principal component
explained_variance_ratio_percentage = pca_all.explained_variance_ratio_ * 100

# Create a figure with one subplot
fig, ax = plt.subplots(figsize=(50, 20))

# Plot the bar plot for explained variance of each principal component
ax.bar(np.arange(1, len(MutCpos_filtered_df.columns) + 1), explained_variance_ratio_percentage, color='skyblue', alpha=0.8)

# Customize the plot
ax.set_xlabel('Principal Component')
ax.set_ylabel('Percentage of Variance Explained')
ax.set_title('Percentage of Variance Explained by Principal Components')
ax.set_xticks(np.arange(1, len(MutCpos_filtered_df.columns) + 1))
#ax.grid(True)

plt.tight_layout()
plt.savefig('MutC_BL_DrugA_Eigen_values_Bar_graph.png')

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutCpos_filtered_df.copy()

# Identify MutCpos groups
baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
drugA_cols = [col for col in Data_csf.columns if "MutCpos-DrugA-12months" in col]

# Custom labels
custom_labels = {
    col: f'MutCpos_BL{i+1}' if col in baseline_cols else f'DRUGA_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Color coding
column_colors = {
    col: 'MediumVioletRed' if col in baseline_cols else 'lightpink'
    for col in Data_csf.columns
}

# For legend
selected_legend_columns = {
    baseline_cols[0]: 'MutCpos Baseline',
    drugA_cols[0]: 'MutCpos DrugA 12mo'
}

selected_columns_list = [baseline_cols, drugA_cols]
centroid_colors = {
    baseline_cols[0]: 'MediumVioletRed',
    drugA_cols[0]: 'lightpink'
}
outline_colors = ['MediumVioletRed', 'lightpink']

# Standardize and transpose
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of MutCpos CSF Samples: Baseline vs. DrugA 12 Months')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutCpos_PCA_PC1_vs_PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutCpos_filtered_df.copy()

# Identify MutCpos groups
baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
drugA_cols = [col for col in Data_csf.columns if "MutCpos-DrugA-12months" in col]

# Custom labels
custom_labels = {
    col: f'MutCpos_BL{i+1}' if col in baseline_cols else f'DRUGA_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Color coding
column_colors = {
    col: 'MediumVioletRed' if col in baseline_cols else 'lightpink'
    for col in Data_csf.columns
}

# For legend
selected_legend_columns = {
    baseline_cols[0]: 'MutCpos Baseline',
    drugA_cols[0]: 'MutCpos DrugA 12mo'
}

selected_columns_list = [baseline_cols, drugA_cols]
centroid_colors = {
    baseline_cols[0]: 'MediumVioletRed',
    drugA_cols[0]: 'lightpink'
}
outline_colors = ['MediumVioletRed', 'lightpink']

# Standardize and transpose
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
    #         weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                     #x facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of MutCpos CSF Samples: Baseline vs. DrugA 12 Months')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutCpos_PCA_PC1_vs_PC2_centroids.png')
plt.show()

In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
pc_loadings_df = pd.DataFrame(pc_loadings.T, columns=['PC1', 'PC2'], index=MutCpos_filtered_df.index)

# Save to CSV
#pc_loadings_df.to_csv('pc_loadings.csv')

# Display the loadings
pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc1_sorted_df = pd.DataFrame(pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
pc1_loading_df_sorted = pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
#pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(pc1_loading_df_sorted.index, pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutC DrugA All PC1 loadings.png')
plt.show()


In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc2_sorted_df = pd.DataFrame(pc_loadings_df['PC2'])

# Sort genes by their PC1 loading in descending order
pc2_loading_df_sorted = pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC1 loadings to CSV
#pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc2_loading_df_sorted

In [ ]:
# Define custom colormap for PC2
custom_cmap_pc2 = mcolors.TwoSlopeNorm(vmin=pc2_loading_df_sorted['PC2'].min(),
                                       vcenter=0,
                                       vmax=pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(30, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap_pc2(pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(pc2_loading_df_sorted.index, pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc2)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutC DrugA All PC2 loadings.png')
plt.show()

In [ ]:
# List of desired gene/protein names
target_names = ['UCHL1', 'GOT1', 'NEFL', 'NEFH', 'NPY', 'AÎ²42']

# Filter the DataFrame
MutC_DrugA_subset_df = MutCpos_filtered_df.loc[MutCpos_filtered_df.index.isin(target_names)]
MutC_DrugA_subset_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutC_DrugA_subset_df.copy()

# Identify MutCpos groups
baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
drugA_cols = [col for col in Data_csf.columns if "MutCpos-DrugA-12months" in col]

# Custom labels
custom_labels = {
    col: f'MutCpos_BL{i+1}' if col in baseline_cols else f'DRUGA_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Color coding
column_colors = {
    col: 'MediumVioletRed' if col in baseline_cols else 'lightpink'
    for col in Data_csf.columns
}

# For legend
selected_legend_columns = {
    baseline_cols[0]: 'MutCpos Baseline',
    drugA_cols[0]: 'MutCpos DrugA 12mo'
}

selected_columns_list = [baseline_cols, drugA_cols]
centroid_colors = {
    baseline_cols[0]: 'MediumVioletRed',
    drugA_cols[0]: 'lightpink'
}
outline_colors = ['MediumVioletRed', 'lightpink']

# Standardize and transpose
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of MutCpos CSF Samples: Baseline vs. DrugA 12 Months')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutCpos__Sigs_prot_PCA_PC1_vs_PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutC_DrugA_subset_df.copy()

# Identify MutCpos groups
baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
drugA_cols = [col for col in Data_csf.columns if "MutCpos-DrugA-12months" in col]

# Custom labels
custom_labels = {
    col: f'MutCpos_BL{i+1}' if col in baseline_cols else f'DRUGA_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Color coding
column_colors = {
    col: 'MediumVioletRed' if col in baseline_cols else 'lightpink'
    for col in Data_csf.columns
}

# For legend
selected_legend_columns = {
    baseline_cols[0]: 'MutCpos Baseline',
    drugA_cols[0]: 'MutCpos DrugA 12mo'
}

selected_columns_list = [baseline_cols, drugA_cols]
centroid_colors = {
    baseline_cols[0]: 'MediumVioletRed',
    drugA_cols[0]: 'lightpink'
}
outline_colors = ['MediumVioletRed', 'lightpink']

# Standardize and transpose
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
       #      weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
  #  ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
   #                   facecolor='none', linewidth=2)
  #  plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of MutCpos CSF Samples: Baseline vs. DrugA 12 Months')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutCpos__Sigs_prot_PCA_PC1_vs_PC2_centroids.png')
plt.show()

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
MutBpos_filtered_df = Data.loc[:, ~Data.columns.str.contains('NoMut|MutC|DrugA-12months|DrugB')]
MutBpos_filtered_df

In [ ]:
MutBpos_filtered_df.columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Copy data
Data_csf = MutBpos_filtered_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else 'gray'
    for col in Data_csf.columns
}

# Legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Groups for centroid and ellipse
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D' if col in mutB_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final plot adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of CSF Samples (MutBpos Combined)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_MutBpos_Combined.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Copy data
Data_csf = MutBpos_filtered_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else 'gray'
    for col in Data_csf.columns
}

# Legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Groups for centroid and ellipse
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D' if col in mutB_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
    #         weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final plot adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of CSF Samples (MutBpos Combined)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_MutBpos_Centriods.png')
plt.show()

In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
pc_loadings_df = pd.DataFrame(pc_loadings.T, columns=['PC1', 'PC2'], index=MutCpos_filtered_df.index)

# Save to CSV
#pc_loadings_df.to_csv('pc_loadings.csv')

# Display the loadings
pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc1_sorted_df = pd.DataFrame(pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
pc1_loading_df_sorted = pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
#pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(pc1_loading_df_sorted.index, pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutB All PC1 loadings.png')
plt.show()

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc2_sorted_df = pd.DataFrame(pc_loadings_df['PC2'])

# Sort genes by their PC1 loading in descending order
pc2_loading_df_sorted = pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC1 loadings to CSV
#pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc2_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc2_loading_df_sorted['PC2'].min(),
                                   vcenter=0,
                                   vmax=pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(pc2_loading_df_sorted.index, pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutB All PC2 loadings.png')
plt.show()

In [ ]:
# List of desired gene/protein names
target_names = ['UCHL1', 'GOT1', 'NEFL', 'NEFH', 'NPY', 'AÎ²42']

MutB_Control_subset_df = MutBpos_filtered_df.loc[MutBpos_filtered_df.index.isin(target_names)]
MutB_Control_subset_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Copy data
Data_csf = MutB_Control_subset_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else 'gray'
    for col in Data_csf.columns
}

# Legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Groups for centroid and ellipse
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D' if col in mutB_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final plot adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of CSF Samples (MutBpos Combined)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_MutBpos_6_Prots.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Copy data
Data_csf = MutB_Control_subset_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else 'gray'
    for col in Data_csf.columns
}

# Legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    control_cols[0]: 'Control',
}

# Groups for centroid and ellipse
selected_columns_list = [mutB_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D' if col in mutB_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
     #        weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final plot adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of CSF Samples (MutBpos Combined)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_MutBpos_6_Prots_Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import euclidean

# ---------- PCA 1: Full Dataset (MutBpos_filtered_df) ----------
Data_csf_1 = MutBpos_filtered_df.copy()
control_cols_1 = [col for col in Data_csf_1.columns if "Control" in col]
mutB_cols_1 = [col for col in Data_csf_1.columns if "MutBpos" in col or "MutB-Baseline" in col]

scaler_1 = StandardScaler()
standardized_data_1 = scaler_1.fit_transform(Data_csf_1)
transposed_data_1 = standardized_data_1.T

pca_1 = PCA(n_components=2)
pca_result_1 = pca_1.fit_transform(transposed_data_1)

indices_control_1 = [Data_csf_1.columns.get_loc(c) for c in control_cols_1]
indices_mutB_1 = [Data_csf_1.columns.get_loc(c) for c in mutB_cols_1]

control_data_1 = pca_result_1[indices_control_1]
mutB_data_1 = pca_result_1[indices_mutB_1]

centroid_control_1 = KMeans(n_clusters=1, random_state=42).fit(control_data_1).cluster_centers_[0]
centroid_mutB_1 = KMeans(n_clusters=1, random_state=42).fit(mutB_data_1).cluster_centers_[0]

distance_1 = euclidean(centroid_control_1, centroid_mutB_1)

# ---------- PCA 2: Subset Dataset (MutB_Control_subset_df) ----------
Data_csf_2 = MutB_Control_subset_df.copy()
control_cols_2 = [col for col in Data_csf_2.columns if "Control" in col]
mutB_cols_2 = [col for col in Data_csf_2.columns if "MutBpos" in col or "MutB-Baseline" in col]

scaler_2 = StandardScaler()
standardized_data_2 = scaler_2.fit_transform(Data_csf_2)
transposed_data_2 = standardized_data_2.T

pca_2 = PCA(n_components=2)
pca_result_2 = pca_2.fit_transform(transposed_data_2)

indices_control_2 = [Data_csf_2.columns.get_loc(c) for c in control_cols_2]
indices_mutB_2 = [Data_csf_2.columns.get_loc(c) for c in mutB_cols_2]

control_data_2 = pca_result_2[indices_control_2]
mutB_data_2 = pca_result_2[indices_mutB_2]

centroid_control_2 = KMeans(n_clusters=1, random_state=42).fit(control_data_2).cluster_centers_[0]
centroid_mutB_2 = KMeans(n_clusters=1, random_state=42).fit(mutB_data_2).cluster_centers_[0]

distance_2 = euclidean(centroid_control_2, centroid_mutB_2)

# ---------- Plotting Distance Comparison ----------
labels = ['Full Dataset (PCA 1)', 'Subset Dataset (PCA 2)']
distances = [distance_1, distance_2]

plt.figure(figsize=(5, 5))
bars = plt.bar(labels, distances, color=['#800080', 'DimGray'])
plt.ylabel('Distance Between Centroids')
plt.title('Centroid Separation in PCA Space (MutBpos vs Control)')

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center')

plt.tight_layout()
plt.savefig('MutBpos_Centroid_Comparison_Barplot.png')
plt.show()

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
NoMut_filtered_df = Data.loc[:, ~Data.columns.str.contains('MutB|MutC|DrugA-12months|DrugB')]
NoMut_filtered_df

In [ ]:
NoMut_filtered_df.columns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Use your filtered dataframe
Data_csf = NoMut_filtered_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
case_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'SALS_{i+1}' if col in case_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in case_cols else 'gray'
    for col in Data_csf.columns
}

# Legend entries
selected_legend_columns = {
    control_cols[0]: 'Control',
    case_cols[0]: 'NoMut Case'
}

# Define groups for centroid and ellipse
selected_columns_list = [control_cols, case_cols]

# Centroid and ellipse colors
centroid_colors = {
    control_cols[0]: 'DimGray',
    case_cols[0]: 'green'
}

outline_colors = ['DimGray', 'green']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else '^' if col in case_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else '^'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of CSF Samples (NoMut Case vs Control)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_NoMutALS_vs_Control.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Use your filtered dataframe
Data_csf = NoMut_filtered_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
case_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'SALS_{i+1}' if col in case_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in case_cols else 'gray'
    for col in Data_csf.columns
}

# Legend entries
selected_legend_columns = {
    control_cols[0]: 'Control',
    case_cols[0]: 'NoMut Case'
}

# Define groups for centroid and ellipse
selected_columns_list = [control_cols, case_cols]

# Centroid and ellipse colors
centroid_colors = {
    control_cols[0]: 'DimGray',
    case_cols[0]: 'green'
}

outline_colors = ['DimGray', 'green']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D' if col in case_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
   # plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
          #   weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                  #    facecolor='none', linewidth=2)
   # plt.gca().add_patch(ellipse)

# Final adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of CSF Samples (NoMut Case vs Control)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_NoMutALS_vs_Control_centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Use your filtered dataframe
Data_csf = NoMut_filtered_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
case_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'SALS_{i+1}' if col in case_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in case_cols else 'gray'
    for col in Data_csf.columns
}

# Legend entries
selected_legend_columns = {
    control_cols[0]: 'Control',
    case_cols[0]: 'NoMut Case'
}

# Define groups for centroid and ellipse
selected_columns_list = [control_cols, case_cols]

# Centroid and ellipse colors
centroid_colors = {
    control_cols[0]: 'DimGray',
    case_cols[0]: 'green'
}

outline_colors = ['DimGray', 'green']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA with at least 3 components
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Plotting PC1 vs PC3
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else '^' if col in case_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else '^'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses for PC1 vs PC3
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # Use PC1 and PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final plot adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of CSF Samples (NoMut Case vs Control) - PC1 vs PC3')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_NoMutALS_vs_Control_PC1_vs_PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Use your filtered dataframe
Data_csf = NoMut_filtered_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
case_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'SALS_{i+1}' if col in case_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in case_cols else 'gray'
    for col in Data_csf.columns
}

# Legend entries
selected_legend_columns = {
    control_cols[0]: 'Control',
    case_cols[0]: 'NoMut Case'
}

# Define groups for centroid and ellipse
selected_columns_list = [control_cols, case_cols]

# Centroid and ellipse colors
centroid_colors = {
    control_cols[0]: 'DimGray',
    case_cols[0]: 'green'
}

outline_colors = ['DimGray', 'green']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA with at least 3 components
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Plotting PC1 vs PC3
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D' if col in case_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
     #        weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses for PC1 vs PC3
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # Use PC1 and PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
     #                 facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final plot adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of CSF Samples (NoMut Case vs Control) - PC1 vs PC3')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_NoMutALS_vs_Control_PC1_vs_PC3_Centroids.png')
plt.show()

In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
pc_loadings_df = pd.DataFrame(pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=MutCpos_filtered_df.index)

# Save to CSV
#pc_loadings_df.to_csv('pc_loadings.csv')

# Display the loadings
pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc1_sorted_df = pd.DataFrame(pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
pc1_loading_df_sorted = pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
#pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(pc1_loading_df_sorted.index, pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('NoMut All PC1 loadings.png')
plt.show()

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc2_sorted_df = pd.DataFrame(pc_loadings_df['PC2'])

# Sort genes by their PC1 loading in descending order
pc2_loading_df_sorted = pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC1 loadings to CSV
#pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc2_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc2_loading_df_sorted['PC2'].min(),
                                   vcenter=0,
                                   vmax=pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors = plt.cm.RdBu_r(custom_cmap(pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(pc2_loading_df_sorted.index, pc2_loading_df_sorted['PC2'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('NoMut All PC2 loadings.png')
plt.show()

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc3_sorted_df = pd.DataFrame(pc_loadings_df['PC3'])

# Sort genes by their PC1 loading in descending order
pc3_loading_df_sorted = pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Save the sorted PC1 loadings to CSV
#pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc3_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap normalization for PC3
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc3_loading_df_sorted['PC3'].min(),
                                   vcenter=0,
                                   vmax=pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors = plt.cm.RdBu_r(custom_cmap(pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(pc3_loading_df_sorted.index, pc3_loading_df_sorted['PC3'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins')

# Properly set x-ticks and labels
ax.set_xticks(range(len(pc3_loading_df_sorted.index)))
ax.set_xticklabels(pc3_loading_df_sorted.index, rotation=90)

# Layout and save
plt.tight_layout()
plt.savefig('NoMut_All_PC3_Loadings.png')
plt.show()

In [ ]:
# List of desired gene/protein names
target_names = ['UCHL1', 'GOT1', 'NEFL', 'NEFH', 'NPY', 'AÎ²42']

NoMut_Control_subset_df = NoMut_filtered_df.loc[NoMut_filtered_df.index.isin(target_names)]
NoMut_Control_subset_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Use your filtered dataframe
Data_csf = NoMut_Control_subset_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
case_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'SALS_{i+1}' if col in case_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in case_cols else 'gray'
    for col in Data_csf.columns
}

# Legend entries
selected_legend_columns = {
    control_cols[0]: 'Control',
    case_cols[0]: 'NoMut Case'
}

# Define groups for centroid and ellipse
selected_columns_list = [control_cols, case_cols]

# Centroid and ellipse colors
centroid_colors = {
    control_cols[0]: 'DimGray',
    case_cols[0]: 'green'
}

outline_colors = ['DimGray', 'green']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else '^' if col in case_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else '^'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of CSF Samples (NoMut Case vs Control)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_NoMutALS_vs_Control_6_Prots.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Use your filtered dataframe
Data_csf = NoMut_Control_subset_df.copy()

# Define groups
control_cols = [col for col in Data_csf.columns if "Control" in col]
case_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'SALS_{i+1}' if col in case_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Assign group colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         'green' if col in case_cols else 'gray'
    for col in Data_csf.columns
}

# Legend entries
selected_legend_columns = {
    control_cols[0]: 'Control',
    case_cols[0]: 'NoMut Case'
}

# Define groups for centroid and ellipse
selected_columns_list = [control_cols, case_cols]

# Centroid and ellipse colors
centroid_colors = {
    control_cols[0]: 'DimGray',
    case_cols[0]: 'green'
}

outline_colors = ['DimGray', 'green']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D' if col in case_cols else 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
          #   weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 'o' if col in control_cols else 'D'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final adjustments
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of CSF Samples (NoMut Case vs Control)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('PCA_CSF_NoMutALS_vs_Control_6_Prots_Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import euclidean

# ---------- PCA 1: Full Dataset ----------
Data_csf_1 = NoMut_filtered_df.copy()
control_cols_1 = [col for col in Data_csf_1.columns if "Control" in col]
case_cols_1 = [col for col in Data_csf_1.columns if "NoMut Case" in col]

scaler_1 = StandardScaler()
standardized_data_1 = scaler_1.fit_transform(Data_csf_1)
transposed_data_1 = standardized_data_1.T

pca_1 = PCA(n_components=2)
pca_result_1 = pca_1.fit_transform(transposed_data_1)

# Get centroids for Control and Case in PCA 1
indices_control_1 = [Data_csf_1.columns.get_loc(c) for c in control_cols_1]
indices_case_1 = [Data_csf_1.columns.get_loc(c) for c in case_cols_1]

control_data_1 = pca_result_1[indices_control_1]
case_data_1 = pca_result_1[indices_case_1]

centroid_control_1 = KMeans(n_clusters=1, random_state=42).fit(control_data_1).cluster_centers_[0]
centroid_case_1 = KMeans(n_clusters=1, random_state=42).fit(case_data_1).cluster_centers_[0]

distance_1 = euclidean(centroid_control_1, centroid_case_1)

# ---------- PCA 2: Subset Dataset ----------
Data_csf_2 = NoMut_Control_subset_df.copy()
control_cols_2 = [col for col in Data_csf_2.columns if "Control" in col]
case_cols_2 = [col for col in Data_csf_2.columns if "NoMut Case" in col]

scaler_2 = StandardScaler()
standardized_data_2 = scaler_2.fit_transform(Data_csf_2)
transposed_data_2 = standardized_data_2.T

pca_2 = PCA(n_components=2)
pca_result_2 = pca_2.fit_transform(transposed_data_2)

# Get centroids for Control and Case in PCA 2
indices_control_2 = [Data_csf_2.columns.get_loc(c) for c in control_cols_2]
indices_case_2 = [Data_csf_2.columns.get_loc(c) for c in case_cols_2]

control_data_2 = pca_result_2[indices_control_2]
case_data_2 = pca_result_2[indices_case_2]

centroid_control_2 = KMeans(n_clusters=1, random_state=42).fit(control_data_2).cluster_centers_[0]
centroid_case_2 = KMeans(n_clusters=1, random_state=42).fit(case_data_2).cluster_centers_[0]

distance_2 = euclidean(centroid_control_2, centroid_case_2)

# ---------- Plotting Distance Comparison ----------
labels = ['Full Dataset (PCA 1)', 'Subset Dataset (PCA 2)']
distances = [distance_1, distance_2]

plt.figure(figsize=(5, 5))
bars = plt.bar(labels, distances, color=['#888', '#1f77b4'])
plt.ylabel('Distance Between Centroids')
plt.title('Centroid Separation in PCA Spaces')

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center')

plt.tight_layout()
plt.savefig('NoMut_Centroid_Comparison_Barplot.png')
plt.show()

In [ ]:
# Remove columns with 'DrugA-12months' or 'DrugB-4month' in their names
MutB_NoMut_filtered_df = Data.loc[:, ~Data.columns.str.contains('MutC|DrugB')]
MutB_NoMut_filtered_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutB_NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_NoMut_PC1_VS_PC2.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutB_NoMut_filtered_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
       #      weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
     #                 facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_NoMut_PC1_VS_PC2_Centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Copy your DataFrame
Data_csf = MutB_NoMut_filtered_df.copy()

# Define group membership
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Groups for ellipses
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA with at least 3 components
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Plot setup
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

# Scatter plot with annotations
for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # PC1 and PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples (PC1 vs PC3) with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_NoMut_PC1_VS_PC3.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# Copy your DataFrame
Data_csf = MutB_NoMut_filtered_df.copy()

# Define group membership
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Colors
column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Groups for ellipses
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA with at least 3 components
pca = PCA(n_components=3)
pca_result = pca.fit_transform(transposed_data)

# Plot setup
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

# Scatter plot with annotations
for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 2]+0.1, custom_labels[col], fontsize=7,
    #         weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 2], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices][:, [0, 2]]  # PC1 and PC3
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.2f}%)')
plt.title('PCA of Case CSF Samples (PC1 vs PC3) with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_NoMut_PC1_VS_PC3_Centroids.png')
plt.show()

In [ ]:
# Extract PC loadings (components show how much each gene contributes to each PC)
pc_loadings = pca.components_

# Create DataFrame: genes as index, PCs as columns
pc_loadings_df = pd.DataFrame(pc_loadings.T, columns=['PC1', 'PC2', 'PC3'], index=MutCpos_filtered_df.index)

# Save to CSV
#pc_loadings_df.to_csv('pc_loadings.csv')

# Display the loadings
pc_loadings_df

In [ ]:
# Extract only PC1 loadings from the full loadings DataFrame
pc1_sorted_df = pd.DataFrame(pc_loadings_df['PC1'])

# Sort genes by their PC1 loading in descending order
pc1_loading_df_sorted = pc1_sorted_df.sort_values(by='PC1', ascending=False)

# Save the sorted PC1 loadings to CSV
pc1_loading_df_sorted.to_csv('pc1_loading_df_sorted.csv')

# Display the sorted DataFrame
pc1_loading_df_sorted

In [ ]:
import matplotlib.colors as mcolors

# Define custom colormap
custom_cmap = mcolors.TwoSlopeNorm(vmin=pc1_loading_df_sorted['PC1'].min(),
                                   vcenter=0,
                                   vmax=pc1_loading_df_sorted['PC1'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC1 values
colors = plt.cm.RdBu_r(custom_cmap(pc1_loading_df_sorted['PC1']))

# Create the bar plot
bars = ax.bar(pc1_loading_df_sorted.index, pc1_loading_df_sorted['PC1'], color=colors)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC1 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC1 Loadings')
ax.set_title('PC1 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc1_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos NoMut All PC1 loadings.png')
plt.show()

In [ ]:
# Extract only PC2 loadings from the full loadings DataFrame
pc2_sorted_df = pd.DataFrame(pc_loadings_df['PC2'])

# Sort genes by their PC2 loading in descending order
pc2_loading_df_sorted = pc2_sorted_df.sort_values(by='PC2', ascending=False)

# Save the sorted PC2 loadings to CSV
pc2_loading_df_sorted.to_csv('pc2_loading_df_sorted.csv')

# Define custom colormap for PC2
custom_cmap_pc2 = mcolors.TwoSlopeNorm(vmin=pc2_loading_df_sorted['PC2'].min(),
                                       vcenter=0,
                                       vmax=pc2_loading_df_sorted['PC2'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC2 values
colors_pc2 = plt.cm.RdBu_r(custom_cmap_pc2(pc2_loading_df_sorted['PC2']))

# Create the bar plot
bars = ax.bar(pc2_loading_df_sorted.index, pc2_loading_df_sorted['PC2'], color=colors_pc2)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc2)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC2 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC2 Loadings')
ax.set_title('PC2 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc2_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos NoMut All PC2 loadings.png')
plt.show()

In [ ]:
# Extract only PC3 loadings from the full loadings DataFrame
pc3_sorted_df = pd.DataFrame(pc_loadings_df['PC3'])

# Sort genes by their PC3 loading in descending order
pc3_loading_df_sorted = pc3_sorted_df.sort_values(by='PC3', ascending=False)

# Save the sorted PC3 loadings to CSV
pc3_loading_df_sorted.to_csv('pc3_loading_df_sorted.csv')

# Define custom colormap for PC3
custom_cmap_pc3 = mcolors.TwoSlopeNorm(vmin=pc3_loading_df_sorted['PC3'].min(),
                                       vcenter=0,
                                       vmax=pc3_loading_df_sorted['PC3'].max())

# Set up the plotting environment
fig, ax = plt.subplots(figsize=(25, 6))

# Define colors for the bars based on the PC3 values
colors_pc3 = plt.cm.RdBu_r(custom_cmap_pc3(pc3_loading_df_sorted['PC3']))

# Create the bar plot
bars = ax.bar(pc3_loading_df_sorted.index, pc3_loading_df_sorted['PC3'], color=colors_pc3)

# Add color bar
sm = plt.cm.ScalarMappable(cmap=plt.cm.RdBu_r, norm=custom_cmap_pc3)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label('PC3 Loadings')

# Set labels and title
ax.set_xlabel('Proteins')
ax.set_ylabel('PC3 Loadings')
ax.set_title('PC3 Loadings for Proteins')

# Rotate gene names for better readability
ax.set_xticklabels(pc3_loading_df_sorted.index, rotation=90)

# Show the plot
plt.tight_layout()
plt.savefig('MutBpos NoMut All PC3 loadings.png')
plt.show()

In [ ]:
# List of desired gene/protein names
target_names = ['UCHL1', 'GOT1', 'NEFL', 'NEFH', 'NPY', 'AÎ²42']

MutBpos_NoMut_Control_subset_df = MutB_NoMut_filtered_df.loc[MutB_NoMut_filtered_df.index.isin(target_names)]
MutBpos_NoMut_Control_subset_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutBpos_NoMut_Control_subset_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_NoMut_PC1_VS_PC2_6_prots.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutBpos_NoMut_Control_subset_df.copy()

# Define group membership based on naming
control_cols = [col for col in Data_csf.columns if "Control" in col]
mutB_cols = [col for col in Data_csf.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols = [col for col in Data_csf.columns if "NoMut Case" in col]

# Custom labels for annotation
custom_labels = {
    col: f'HC{i+1}' if col in control_cols else
         f'MutB_{i+1}' if col in mutB_cols else
         f'SALS_{i+1}' if col in noMut_cols else f'O_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

column_colors = {
    col: 'DimGray' if col in control_cols else
         '#800080' if col in mutB_cols else
         'green' if col in noMut_cols else 'gray'
    for col in Data_csf.columns
}

# Choose one representative column per group for legend
selected_legend_columns = {
    mutB_cols[0]: 'MutBpos',
    noMut_cols[0]: 'NoMut Case',
    control_cols[0]: 'Control',
}

# Define groups for centroid/ellipse plotting
selected_columns_list = [mutB_cols, noMut_cols, control_cols]

centroid_colors = {
    mutB_cols[0]: '#800080',
    noMut_cols[0]: 'green',
    control_cols[0]: 'DimGray'
}

outline_colors = ['#800080', 'green', 'DimGray']

# Standardize and transpose data
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Start plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5)
    #plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
       #      weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    if col in control_cols:
        marker = 'o'
    elif col in mutB_cols:
        marker = 'D'
    elif col in noMut_cols:
        marker = '^'
    else:
        marker = 'x'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=0,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and confidence ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    #ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
    #                  facecolor='none', linewidth=2)
    #plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of Case CSF Samples with Centroids and Confidence Ellipses')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('MutBpos_NoMut_PC1_VS_PC2_6_prots_centroids.png')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import euclidean

# --------- PCA 1: Full Dataset (MutBpos + NoMut + Controls) ---------
Data_csf_1 = MutB_NoMut_filtered_df.copy()

control_cols_1 = [col for col in Data_csf_1.columns if "Control" in col]
mutB_cols_1 = [col for col in Data_csf_1.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols_1 = [col for col in Data_csf_1.columns if "NoMut Case" in col]

scaler_1 = StandardScaler()
standardized_data_1 = scaler_1.fit_transform(Data_csf_1)
transposed_data_1 = standardized_data_1.T

pca_1 = PCA(n_components=2)
pca_result_1 = pca_1.fit_transform(transposed_data_1)

# Get centroids
indices_control_1 = [Data_csf_1.columns.get_loc(c) for c in control_cols_1]
indices_mutB_1 = [Data_csf_1.columns.get_loc(c) for c in mutB_cols_1]
indices_noMut_1 = [Data_csf_1.columns.get_loc(c) for c in noMut_cols_1]

centroid_control_1 = KMeans(n_clusters=1, random_state=42).fit(pca_result_1[indices_control_1]).cluster_centers_[0]
centroid_mutB_1 = KMeans(n_clusters=1, random_state=42).fit(pca_result_1[indices_mutB_1]).cluster_centers_[0]
centroid_noMut_1 = KMeans(n_clusters=1, random_state=42).fit(pca_result_1[indices_noMut_1]).cluster_centers_[0]

distance_mutB_control_1 = euclidean(centroid_mutB_1, centroid_control_1)
distance_noMut_control_1 = euclidean(centroid_noMut_1, centroid_control_1)

# --------- PCA 2: Subset Dataset (reduced proteins/features) ---------
Data_csf_2 = MutBpos_NoMut_Control_subset_df.copy()

control_cols_2 = [col for col in Data_csf_2.columns if "Control" in col]
mutB_cols_2 = [col for col in Data_csf_2.columns if "MutBpos" in col or "MutB-Baseline" in col]
noMut_cols_2 = [col for col in Data_csf_2.columns if "NoMut Case" in col]

scaler_2 = StandardScaler()
standardized_data_2 = scaler_2.fit_transform(Data_csf_2)
transposed_data_2 = standardized_data_2.T

pca_2 = PCA(n_components=2)
pca_result_2 = pca_2.fit_transform(transposed_data_2)

# Get centroids
indices_control_2 = [Data_csf_2.columns.get_loc(c) for c in control_cols_2]
indices_mutB_2 = [Data_csf_2.columns.get_loc(c) for c in mutB_cols_2]
indices_noMut_2 = [Data_csf_2.columns.get_loc(c) for c in noMut_cols_2]

centroid_control_2 = KMeans(n_clusters=1, random_state=42).fit(pca_result_2[indices_control_2]).cluster_centers_[0]
centroid_mutB_2 = KMeans(n_clusters=1, random_state=42).fit(pca_result_2[indices_mutB_2]).cluster_centers_[0]
centroid_noMut_2 = KMeans(n_clusters=1, random_state=42).fit(pca_result_2[indices_noMut_2]).cluster_centers_[0]

distance_mutB_control_2 = euclidean(centroid_mutB_2, centroid_control_2)
distance_noMut_control_2 = euclidean(centroid_noMut_2, centroid_control_2)

# --------- Plotting Distance Comparison ---------
labels = ['MutB vs Control (Full)', 'SALS vs Control (Full)',
          'MutB vs Control (Subset)', 'SALS vs Control (Subset)']
distances = [distance_mutB_control_1, distance_noMut_control_1,
             distance_mutB_control_2, distance_noMut_control_2]

plt.figure(figsize=(7, 5))
bars = plt.bar(labels, distances, color=['#800080', 'green', '#dda0dd', '#90ee90'])
plt.ylabel('Distance Between Centroids')
plt.title('Centroid Separation in PCA Spaces')

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2., height, f'{height:.2f}', ha='center')

plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('MutB_NoMut_Centroid_Distance_Comparison.png')
plt.show()

In [ ]:
df1 = pd.read_excel("Expected_Corr.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df1

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

# Prepare data
x = df1['Delta observed-expected 6']
y = df1['Delta 6-BL pNfH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
from scipy.stats import pearsonr
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Store the plot in a variable
reg_plot = sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Change the color of the confidence interval band
for collection in reg_plot.collections:
    collection.set_facecolor("darkgray")  # Change "lightblue" to your preferred color


# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df1.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL pNfH'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL pNfH')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL pNfH vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()

In [ ]:
df2 = pd.read_excel("Expected_Corr.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df2

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

# Prepare data
x = df2['Delta observed-expected 6']
y = df2['Delta 6-BL NfL']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
from scipy.stats import pearsonr
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df2.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL NfL'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL NfL')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL NfL vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()

In [ ]:
df3 = pd.read_excel("Expected_Corr.xlsx", sheet_name=2, index_col=0)  # 0 refers to the first sheet
df3

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

# Prepare data
x = df3['Delta observed-expected 6']
y = df3['Delta 6-BL UCHL1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
from scipy.stats import pearsonr
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df3.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL UCHL1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL UCHL1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL UCHL1 vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()

In [ ]:
df4 = pd.read_excel("Expected_Corr.xlsx", sheet_name=3, index_col=0)  # 0 refers to the first sheet
df4

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df4['Delta observed-expected 6']
y = df4['Delta 6-BL Aß1-42']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df4.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL Aß1-42'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL Aß1-42')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL Aß1-42 vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()

In [ ]:
df5 = pd.read_excel("Expected_Corr.xlsx", sheet_name=4, index_col=0)  # 0 refers to the first sheet
df5

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df5['Delta observed-expected 6']
y = df5['Delta 6-BL Aß1-42 Males']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df5.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL Aß1-42 Males'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL Aß1-42 Males')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL Aß1-42 Males vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()

In [ ]:
df6 = pd.read_excel("Expected_Corr.xlsx", sheet_name=5, index_col=0)  # 0 refers to the first sheet
df6

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df6['Delta observed-expected 6']
y = df6['Delta 6-BL Aß1-42 Females']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df6.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL Aß1-42 Females'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL Aß1-42 Females')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL Aß1-42 Females vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()

In [ ]:
df7 = pd.read_excel("Expected_Corr.xlsx", sheet_name=6, index_col=0)  # 0 refers to the first sheet
df7

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df7['Delta observed-expected 6']
y = df7['Delta 6-BL Aß1-40']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df7.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL Aß1-40'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL Aß1-40')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL Aß1-40 vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()


In [ ]:
df8 = pd.read_excel("Expected_Corr.xlsx", sheet_name=7, index_col=0)  # 0 refers to the first sheet
df8

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df8['Delta observed-expected 6']
y = df8['Delta 6-BL Aß1-40 Males']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df8.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL Aß1-40 Males'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL Aß1-40 Males')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL Aß1-40 Males vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()

In [ ]:
df9 = pd.read_excel("Expected_Corr.xlsx", sheet_name=8, index_col=0)  # 0 refers to the first sheet
df9

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df9['Delta observed-expected 6']
y = df9['Delta 6-BL Aß1-40 Females']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 6']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 6']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 6']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df9.iterrows():
    plt.text(row['Delta observed-expected 6'], row['Delta 6-BL Aß1-40 Females'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Delta observed-expected 6')
plt.ylabel('Delta 6-BL Aß1-40 Females')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('Delta 6-BL Aß1-40 Females vs Delta observed-expected 6.png', bbox_inches='tight')

plt.show()


In [ ]:
df10 = pd.read_excel("Corr_12months.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df10

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df10['Delta observed-expected 12']
y = df10['NfL']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 12']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 12']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 12']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df10.iterrows():
    plt.text(row['Delta observed-expected 12'], row['NfL'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot ', fontsize=14)
plt.xlabel('Delta observed-expected 12')
plt.ylabel('NfL')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NfL vs Delta observed-expected 12.png', bbox_inches='tight')

plt.show()

In [ ]:
df11 = pd.read_excel("Corr_12months.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df11

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df11['Delta observed-expected 12']
y = df11['NfH']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 12']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 12']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 12']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df11.iterrows():
    plt.text(row['Delta observed-expected 12'], row['NfH'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Delta observed-expected 12')
plt.ylabel('NfH')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NfH vs Delta observed-expected 12.png', bbox_inches='tight')

plt.show()

In [ ]:
df12 = pd.read_excel("Corr_12months.xlsx", sheet_name=2, index_col=0)  # 0 refers to the first sheet
df12

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df12['Delta observed-expected 12']
y = df12['NPTX1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 12']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 12']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 12']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df12.iterrows():
    plt.text(row['Delta observed-expected 12'], row['NPTX1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Delta observed-expected 12')
plt.ylabel('NPTX1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NPTX1 vs Delta observed-expected 12.png', bbox_inches='tight')

plt.show()

In [ ]:
df13 = pd.read_excel("Corr_12months.xlsx", sheet_name=3, index_col=0)  # 0 refers to the first sheet
df13

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df13['Delta observed-expected 12']
y = df13['NPTX2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 12']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 12']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 12']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df13.iterrows():
    plt.text(row['Delta observed-expected 12'], row['NPTX2'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Delta observed-expected 12')
plt.ylabel('NPTX2')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NPTX2 vs Delta observed-expected 12.png', bbox_inches='tight')

plt.show()

In [ ]:
df14 = pd.read_excel("Corr_12months.xlsx", sheet_name=4, index_col=0)  # 0 refers to the first sheet
df14

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df14['Delta observed-expected 12']
y = df14['NPTXR']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 12']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 12']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 12']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df14.iterrows():
    plt.text(row['Delta observed-expected 12'], row['NPTXR'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Delta observed-expected 12')
plt.ylabel('NPTXR')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('NPTXR vs Delta observed-expected 12.png', bbox_inches='tight')

plt.show()

In [ ]:
# List of desired gene/protein names
target_names = ['NPTX1', 'NPTX2', 'NPTXR', 'NEFH', 'NEFL']

# Filter the DataFrame
MutC_DrugA_subset_df_NPTX = MutCpos_filtered_df.loc[MutCpos_filtered_df.index.isin(target_names)]
MutC_DrugA_subset_df_NPTX

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from matplotlib.patches import Ellipse
from scipy.stats import chi2

Data_csf = MutC_DrugA_subset_df_NPTX.copy()

# Identify MutCpos groups
baseline_cols = [col for col in Data_csf.columns if "MutCpos-Baseline" in col]
drugA_cols = [col for col in Data_csf.columns if "MutCpos-DrugA-12months" in col]

# Custom labels
custom_labels = {
    col: f'MutCpos_BL{i+1}' if col in baseline_cols else f'DRUGA_{i+1}'
    for i, col in enumerate(Data_csf.columns)
}

# Color coding
column_colors = {
    col: 'MediumVioletRed' if col in baseline_cols else 'lightpink'
    for col in Data_csf.columns
}

# For legend
selected_legend_columns = {
    baseline_cols[0]: 'MutCpos Baseline',
    drugA_cols[0]: 'MutCpos DrugA 12mo'
}

selected_columns_list = [baseline_cols, drugA_cols]
centroid_colors = {
    baseline_cols[0]: 'MediumVioletRed',
    drugA_cols[0]: 'lightpink'
}
outline_colors = ['MediumVioletRed', 'lightpink']

# Standardize and transpose
scaler = StandardScaler()
standardized_data = scaler.fit_transform(Data_csf)
transposed_data = standardized_data.T

# PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(transposed_data)

# Plotting
sns.set(style='white', rc={'figure.figsize': (11, 8)}, font_scale=1.1)
plt.figure()

for i, col in enumerate(Data_csf.columns):
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5)
    plt.text(pca_result[i, 0]-0.1, pca_result[i, 1]+0.1, custom_labels[col], fontsize=7,
             weight='bold', color='black', ha='center', va='center')

# Legend
for col in selected_legend_columns:
    i = Data_csf.columns.get_loc(col)
    color = column_colors[col]
    marker = 's' if col in baseline_cols else 'p'
    plt.scatter(pca_result[i, 0], pca_result[i, 1], color=color, marker=marker, s=100,
                edgecolor='k', linewidth=0.5, label=selected_legend_columns[col])

# Centroids and ellipses
for group_cols, edge_color in zip(selected_columns_list, outline_colors):
    if not group_cols:
        continue
    indices = [Data_csf.columns.get_loc(c) for c in group_cols]
    group_data = pca_result[indices]
    kmeans = KMeans(n_clusters=1, random_state=42).fit(group_data)
    centroid = kmeans.cluster_centers_[0]
    plt.scatter(*centroid, marker='X', s=200, color=centroid_colors[group_cols[0]],
                edgecolor='k', linewidth=1)

    cov = np.cov(group_data.T)
    eigvals, eigvecs = np.linalg.eig(cov)
    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))
    width, height = 2 * np.sqrt(chi2.ppf(0.95, 2)) * np.sqrt(eigvals)
    ellipse = Ellipse(centroid, width, height, angle=angle, edgecolor=edge_color,
                      facecolor='none', linewidth=2)
    plt.gca().add_patch(ellipse)

# Final touches
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
plt.title('PCA of MutCpos CSF Samples: Baseline vs. DrugA 12 Months')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Groups')
plt.tight_layout()
plt.savefig('NPTX_MutCpos_PCA_PC1_vs_PC2.png')
plt.show()

In [ ]:
df15 = pd.read_excel("NPTX 3months Corr.xlsx", sheet_name=0, index_col=0)  # 0 refers to the first sheet
df15

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df15['Delta observed-expected 3']
y = df15['3-BL NPTX1']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 3']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 3']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 3']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df15.iterrows():
    plt.text(row['Delta observed-expected 3'], row['3-BL NPTX1'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Delta observed-expected 3')
plt.ylabel('3-BL NPTX1')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('3-BL NPTX1 vs Delta observed-expected 3.png', bbox_inches='tight')

plt.show()

In [ ]:
df16 = pd.read_excel("NPTX 3months Corr.xlsx", sheet_name=1, index_col=0)  # 0 refers to the first sheet
df16

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df16['Delta observed-expected 3']
y = df16['3-BL NPTX2']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 3']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 3']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 3']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df16.iterrows():
    plt.text(row['Delta observed-expected 3'], row['3-BL NPTX2'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Delta observed-expected 3')
plt.ylabel('3-BL NPTX2')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('3-BL NPTX2 vs Delta observed-expected 3.png', bbox_inches='tight')

plt.show()

In [ ]:
df17 = pd.read_excel("NPTX 3months Corr.xlsx", sheet_name=2, index_col=0)  # 0 refers to the first sheet
df17

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np
from scipy.stats import pearsonr

# Prepare data
x = df17['Delta observed-expected 3']
y = df17['3-BL NPTXR']
x_with_const = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, x_with_const).fit()
slope_conf_int = model.conf_int().loc['Delta observed-expected 3']
intercept_conf_int = model.conf_int().loc['const']

# Get model values
r_squared = model.rsquared
f_stat = model.fvalue
f_pvalue = model.f_pvalue
slope = model.params['Delta observed-expected 3']
intercept = model.params['const']
slope_se = model.bse['Delta observed-expected 3']
intercept_se = model.bse['const']

# Pearson correlation
pearson_corr, pearson_p = pearsonr(x, y)

# Plotting
plt.figure(figsize=(10, 10))

# Regression line only
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    line_kws={'color': 'black', 'linestyle': '--'},
    ci=95
)

# Triangle scatter points
plt.scatter(x, y, marker='^', s=100, color='mediumvioletred')

# Index labels
for index, row in df17.iterrows():
    plt.text(row['Delta observed-expected 3'], row['3-BL NPTXR'], str(index), fontsize=9, weight='bold')

# Axis titles
plt.title('Scatter Plot', fontsize=14)
plt.xlabel('Delta observed-expected 3')
plt.ylabel('3-BL NPTXR')
plt.grid(False)

# Table data
table_data = [
    ["Pearson r", f"{pearson_corr:.2f}"],
    ["Pearson p-value", f"{pearson_p:.8f}"],
    ["R²", f"{r_squared:.2f}"],
    ["F-statistic", f"{f_stat:.2f}"],
    ["F-test p-value", f"{f_pvalue:.8f}"],
    ["Slope", f"{slope:.2f} ± {slope_se:.2f}"],
    ["Slope 95% CI", f"[{slope_conf_int[0]:.2f}, {slope_conf_int[1]:.2f}]"],
    ["Intercept", f"{intercept:.2f} ± {intercept_se:.2f}"],
    ["Intercept 95% CI", f"[{intercept_conf_int[0]:.2f}, {intercept_conf_int[1]:.2f}]"]
]

# Draw table
plt.table(
    cellText=table_data,
    colLabels=["Statistic", "Value"],
    cellLoc='center',
    loc='bottom',
    bbox=[0.15, -0.6, 0.7, 0.45]
)

plt.subplots_adjust(bottom=0.5)
plt.tight_layout()

plt.savefig('3-BL NPTXR vs Delta observed-expected 3.png', bbox_inches='tight')

plt.show()